# PredictGuard — Phase 1, Stage 2: Target Construction

**Project**: Explainable Predictive Maintenance System  
**Dataset**: Microsoft Azure Predictive Maintenance  
**Author**: PredictGuard Contributors  
**Stage**: 2 of 6 — Target Construction

---

## Objective

Construct **leakage-safe prediction targets** for the predictive maintenance system.  
All computational logic lives in `src/target_creation.py`. This notebook orchestrates the workflow and provides visual verification.

### Prediction Problem

> *"Given machine telemetry at hour **t**, predict whether the machine will fail within the next **24 hours**."*

### Generated Targets
1. `y_failure` (binary: 1 = failure in `(t, t+24h]`, 0 = otherwise)
2. `failure_component` (multiclass: `comp1`, `comp2`, `comp3`, `comp4`, or `NaN` for `y_failure=0`)
3. `time_to_failure_hours` (auxiliary: hours until first failure in window)

---
## 0. Environment Setup & Imports

In [ ]:
import logging
import sys
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Ensure src/ is importable
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import target_creation as tc

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("stage2_notebook")
logger.info("Stage 2 notebook started.")

In [ ]:
RAW_DIR = project_root / "data" / "raw"
PROCESSED_DIR = project_root / "data" / "processed"
REPORTS_DIR = project_root / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

---
## 1. Load Cleaned Data

In [ ]:
datasets = tc.load_stage1_data(RAW_DIR)
telemetry = datasets["telemetry"]
failures = datasets["failures"]

print(f"Telemetry shape: {telemetry.shape}")
print(f"Failures shape:  {failures.shape}")

---
## 2. Validate Inputs

In [ ]:
issues = tc.validate_inputs(telemetry, failures)
if not issues:
    print("✅ Input validation passed successfully.")
else:
    print(f"❌ Found {len(issues)} issues:")
    for issue in issues:
        print(f"  • {issue}")

---
## 3. Construct Binary Target (`y_failure`)

> **Leakage Prevention Strategy**:
> Each machine is processed independently using `groupby(machineID)`. Future windows are computed strictly per machine. No global shifts or cross-machine timestamp matching are permitted.

In [ ]:
df_labelled = tc.compute_binary_target(telemetry, failures, horizon_hours=24)
display(df_labelled.head(5))

---
## 4. Assign Component Targets (`failure_component`)

> For rows where `y_failure=1`, assign the component of the **first failure** occurring in `(t, t+24h]`. Set to `NaN` for `y_failure=0`.

In [ ]:
df_labelled = tc.assign_failure_component(df_labelled, failures, horizon_hours=24)

print("\nSample positive labels:")
display(df_labelled[df_labelled["y_failure"] == 1][["machineID", "datetime", "y_failure", "failure_component", "time_to_failure_hours"]].head(5))

---
## 5. Exhaustive Target Validation & Spot Checks

In [ ]:
val_results = tc.validate_targets(df_labelled, failures, horizon_hours=24, spot_check_n=5, random_seed=42)

print("\n── Validation Summary ──")
print(f"  Positive check passed: {val_results['positive_check_pass']}")
print(f"  Negative check passed: {val_results['negative_check_pass']}")
print(f"  Component check passed: {val_results['component_check_pass']}")
print(f"  NaN consistency passed: {val_results['nan_consistency_pass']}")

---
## 6. Compute Statistics & Persist Outputs

In [ ]:
stats = tc.compute_target_statistics(df_labelled, failures)

# Save datasets
tc.save_targets(df_labelled, PROCESSED_DIR)

# Save Markdown Report
tc.generate_target_report(
    df=df_labelled,
    stats=stats,
    validation=val_results,
    output_path=REPORTS_DIR / "target_construction_report.md",
    horizon_hours=24,
)
print("✅ Datasets and Markdown report saved successfully.")

---
## 7. Generate Visualisations

In [ ]:
tc.plot_class_distribution(df_labelled, FIGURES_DIR)
tc.plot_failure_timeline(failures, FIGURES_DIR)
tc.plot_component_frequency(df_labelled, FIGURES_DIR)
tc.plot_positives_per_machine(df_labelled, FIGURES_DIR)
tc.plot_time_to_failure_histogram(df_labelled, FIGURES_DIR, horizon_hours=24)
tc.plot_positive_windows_sample(df_labelled, failures, FIGURES_DIR, horizon_hours=24)
tc.plot_failure_heatmap(failures, df_labelled, FIGURES_DIR)

print("✅ All 7 visualisations generated and saved under reports/figures/.")

---
## 8. Stage 2 Summary & Next Steps

| Deliverable | Status |
|---|---|
| Binary target (`y_failure`) | ✅ Constructed & Validated |
| Component target (`failure_component`) | ✅ Assigned |
| Leakage check | ✅ 0 leakage violations |
| Parquet & CSV outputs | ✅ Saved under `data/processed/` |
| Markdown report | ✅ Saved to `reports/target_construction_report.md` |
| 7 Figures | ✅ Saved to `reports/figures/` |

### Ready for Stage 3 (Feature Engineering)